<a href="https://colab.research.google.com/github/nawab-khan/ovs-thesis/blob/main/notebooks/day1_grounded_sam_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

x = torch.randn(2, 2)

print(x)
print("torch ok")

2.12.0.dev20260408+cu128
True
tensor([[ 0.3278,  0.6308],
        [-0.5657, -0.8507]])
torch ok


In [2]:
from PIL import Image
import requests

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

image_pil = Image.open(
    requests.get(image_url, stream=True).raw
).convert("RGB")

print(image_pil.size)
print("image ok")

(640, 480)
image ok


In [3]:
from segment_anything import sam_model_registry, SamPredictor

print("sam import ok")

sam import ok


In [4]:
SAM_CHECKPOINT = "../checkpoints/sam_vit_b_01ec64.pth"

sam = sam_model_registry["vit_b"](
    checkpoint=SAM_CHECKPOINT
)

print("sam loaded")

sam loaded


In [5]:
from transformers import AutoProcessor
from transformers import AutoModelForZeroShotObjectDetection

print("imports ok")

C:\Users\Admin\anaconda3\envs\ovs-thesis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports ok


In [6]:
DEVICE = "cpu"

model_id = "IDEA-Research/grounding-dino-tiny"

processor = AutoProcessor.from_pretrained(model_id)

print("processor ok")

processor ok


In [7]:
grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    model_id
)

print("model loaded")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 990/990 [00:00<00:00, 8425.32it/s]

model loaded


In [8]:
from PIL import Image
import requests

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

image_pil = Image.open(
    requests.get(image_url, stream=True).raw
).convert("RGB")

print(image_pil.size)

(640, 480)


In [9]:
inputs = processor(
    images=image_pil,
    text="cat . remote control . blanket",
    return_tensors="pt"
)

print("inputs ok")

with torch.no_grad():
    outputs = grounding_model(**inputs)

print("forward ok")

inputs ok
forward ok


In [10]:
results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    threshold=0.25,
    text_threshold=0.25,
    target_sizes=[image_pil.size[::-1]]
)

print(results)
print("postprocess ok")

[{'scores': tensor([0.8366, 0.8055, 0.4843, 0.5207]), 'boxes': tensor([[  9.7824,  53.8946, 316.6847, 474.5156],
        [345.6510,  23.7496, 639.1973, 372.9802],
        [333.1038,  75.7954, 370.8618, 187.0376],
        [ 40.0932,  72.2107, 176.1380, 117.6376]]), 'text_labels': ['cat', 'cat', 'remote control', 'remote control'], 'labels': ['cat', 'cat', 'remote control', 'remote control']}]
postprocess ok


In [11]:
result = results[0]

boxes = result["boxes"].cpu().numpy()

print(boxes.shape)
print(boxes)

(4, 4)
[[  9.782448  53.894592 316.68472  474.5156  ]
 [345.65097   23.749596 639.1973   372.98022 ]
 [333.10382   75.79537  370.8618   187.0376  ]
 [ 40.093227  72.210686 176.138    117.63755 ]]


In [12]:
from segment_anything import sam_model_registry, SamPredictor

SAM_CHECKPOINT = "../checkpoints/sam_vit_b_01ec64.pth"

sam = sam_model_registry["vit_b"](
    checkpoint=SAM_CHECKPOINT
)

sam.to(device="cpu")

predictor = SamPredictor(sam)

print("predictor ready")

predictor ready


In [13]:
import numpy as np

image_np = np.array(image_pil)

predictor.set_image(image_np)

print("image set ok")

image set ok


In [14]:
box = boxes[0]

mask, score, logits = predictor.predict(
    point_coords=None,
    point_labels=None,
    box=box,
    multimask_output=False
)

print(mask.shape)
print("sam predict ok")

(1, 480, 640)
sam predict ok


In [15]:
import cv2
import numpy as np

overlay = image_np.copy()

overlay[mask[0] > 0] = [255, 0, 0]

overlay_bgr = cv2.cvtColor(
    overlay,
    cv2.COLOR_RGB2BGR
)

cv2.imwrite(
    "grounded_sam_result.jpg",
    overlay_bgr
)

print("saved")

saved


In [16]:
all_masks = []

for box in boxes:

    mask, score, logits = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=box,
        multimask_output=False
    )

    all_masks.append(mask[0])

print(f"{len(all_masks)} masks generated")

4 masks generated


In [17]:
overlay = image_np.copy()

colors = [
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255],
    [255, 255, 0]
]

for i, mask in enumerate(all_masks):

    overlay[mask > 0] = colors[i % len(colors)]

overlay_bgr = cv2.cvtColor(
    overlay,
    cv2.COLOR_RGB2BGR
)

cv2.imwrite(
    "grounded_sam_multi.jpg",
    overlay_bgr
)

print("saved")

saved


In [ ]:
import torch
import requests
from transformers import pipeline
from segment_anything import sam_model_registry, SamPredictor
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt
from supervision import BoxAnnotator, MaskAnnotator

print("✓ Done")
# # DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# DEVICE = "cpu"
# print("Using device:", DEVICE)

In [ ]:
import torch

DEVICE = "cpu"

print("Using device:", DEVICE)

if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

# SAM_CHECKPOINT = "../checkpoints/sam_vit_h_4b8939.pth"

# sam = sam_model_registry["vit_h"](
#     checkpoint=SAM_CHECKPOINT
# )

SAM_CHECKPOINT = "../checkpoints/sam_vit_b_01ec64.pth"

sam = sam_model_registry["vit_b"](
    checkpoint=SAM_CHECKPOINT
)

sam.to(device=DEVICE)

predictor = SamPredictor(sam)

print("✓ SAM loaded")

In [ ]:
from transformers import AutoProcessor
from transformers import AutoModelForZeroShotObjectDetection

model_id = "IDEA-Research/grounding-dino-tiny"

processor = AutoProcessor.from_pretrained(model_id)

grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(DEVICE)

print("✓ HF GroundingDINO loaded")

In [ ]:
# # =============================================================================
# REPLACE your GroundedSAMPipeline class WITH THIS:
# =============================================================================
from torchvision.ops import box_convert
class GroundedSAMPipeline:
    """Modular baseline: TEXT → GroundingDINO → Boxes → SAM → Masks"""

    def __init__(self, model, sam_predictor, device):
        # self.processor = processor
        self.model = model
        self.predictor = sam_predictor
        self.device = device
        self._image_hash = None

    def detect(
        self,
        image_pil,
        text_prompt,
        box_threshold=0.25,
        text_threshold=0.25
    ):

        inputs = processor(
            images=image_pil,
            text=text_prompt,
            return_tensors="pt"
        ).to(self.device)
    
        with torch.no_grad():
    
            outputs = self.model(**inputs)
    
        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=[image_pil.size[::-1]]
        )
    
        result = results[0]
    
        return (
            result["boxes"].cpu().numpy(),
            result["scores"].cpu().numpy(),
            result["text_labels"]
        )
    
    def segment(self, image_np, boxes):
        """SAM segmentation for each box."""
        if boxes.shape[0] == 0:
            return np.empty((0, image_np.shape[0], image_np.shape[1]))

        h, w = image_np.shape[:2]
        curr_hash = hash((h, w, id(image_np)))
        if self._image_hash != curr_hash:
            self.predictor.set_image(image_np)
            self._image_hash = curr_hash

        masks_list = []
        for box in boxes:
            mask, _, _ = self.predictor.predict(
                point_coords=None, point_labels=None,
                box=box.astype(np.float32), multimask_output=False
            )
            mask = mask.squeeze(0) if mask.ndim == 3 else mask
            masks_list.append(mask)

        return np.stack(masks_list, axis=0)

    def run(self, image_pil, text_prompt, box_threshold=0.25):
        """End-to-end pipeline."""
        boxes, scores, labels = self.detect(image_pil, text_prompt, box_threshold)
        image_np = np.array(image_pil.convert("RGB"))  # RGB → BGR
        masks = self.segment(image_np, boxes)

        return {
            "image_np": image_np,
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
            "masks": masks
        }

    def visualize(self, result, show_boxes=True, show_masks=True, figsize=(14, 6)):
        """Visualize detections + masks."""
        # image_rgb = cv2.cvtColor(result["image_np"], cv2.COLOR_BGR2RGB)
        image_rgb = result["image_np"]
        h, w = image_rgb.shape[:2]

        scale = 800 / max(h, w) if max(h, w) > 800 else 1.0
        disp = cv2.resize(image_rgb, (int(w*scale), int(h*scale))) if scale < 1.0 else image_rgb

        fig, axes = plt.subplots(1, 2 if show_masks else 1, figsize=figsize)
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])

        # Boxes
        axes[0].imshow(disp)
        axes[0].set_title("Open-Vocabulary Detections (GroundingDINO)")

        for box, score, label in zip(result["boxes"] * scale, result["scores"], result["labels"]):
            x1, y1, x2, y2 = box.astype(int)
            axes[0].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, edgecolor="cyan", facecolor="none", linewidth=2))
            axes[0].text(x1, y1-5, f"{label} ({score:.2f})", color="cyan", fontsize=9, fontweight="bold",
                        bbox=dict(boxstyle="round", facecolor="cyan", alpha=0.3))
        axes[0].axis("off")

        # Masks
        if show_masks:
            axes[1].imshow(disp)
            axes[1].set_title("Segmentation Masks (SAM)")
            for mask in result["masks"]:
                mask = mask.squeeze(0) if mask.ndim == 3 else mask
                axes[1].imshow(mask, cmap="gray", alpha=0.35)
                axes[1].contour(mask, colors="yellow", linewidths=1)
            axes[1].axis("off")

        plt.tight_layout()
        plt.show()

print("a")


In [ ]:
# # Initialize pipeline
# # pipeline = GroundedSAMPipeline(detector, predictor, DEVICE)

# # NEW initialization:
# pipeline = GroundedSAMPipeline(
#     grounding_model,
#     predictor,
#     DEVICE
# )


# # Load image (URL or local)
# image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# image_pil = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# # Run open-vocabulary instance segmentation
# text_prompt = "cat. remote control. blanket."
# result = pipeline.run(image_pil, text_prompt, box_threshold=0.25)

# print(f"Detected {len(result['labels'])} objects:")
# for label, score in zip(result["labels"], result["scores"]):
#     print(f"  {label}: {score:.3f}")

# # Visualize
# pipeline.visualize(result, show_boxes=True, show_masks=True)


In [ ]:
# Initialize pipeline
pipeline = GroundedSAMPipeline(
    grounding_model,
    predictor,
    DEVICE
)

# Load image
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"

image_pil = Image.open(
    requests.get(image_url, stream=True).raw
).convert("RGB")

# Prompt
text_prompt = "cat . remote control . blanket"

# Run pipeline
result = pipeline.run(
    image_pil,
    text_prompt,
    box_threshold=0.25
)

print(f"Detected {len(result['labels'])} objects:")

for label, score in zip(
    result["labels"],
    result["scores"]
):
    print(f"  {label}: {score:.3f}")

# Visualize
pipeline.visualize(
    result,
    show_boxes=True,
    show_masks=True
)